# Vulnerability indicator 

This notebook loads the sensitivity and lack of adaptive capacity raster layers and aggregates them using simple additive aggregation. The resulting score is normalized. It creates: 
- a raster with mean vulnerability score at a chosen pixel size
- a raster with vulnerability score at a chosen pixel size (not normalized)
- a raster with missing variable counts at a chosen pixel size
- a map figure for vulnerability
- a map figure for known variable counts

## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `sensitivity_mean.tif`
- `lackof_adapt_mean.tif`
- `bii_5000m.tif`
- `World_Countries_(Generalized)_8414823838130214587.gpkg` (or your country layer)



In [ ]:
# Configuration (edit these paths if needed)

SENSITIVITY = 'sensitivity\\sensitivity_mean.tif'
LACKOF_ADAPT = 'lackof_adapt\\lackof_adapt_mean.tif'
REF= 'sensitivity\\biodiversity_intactness\\bii_5000m.tif' 
WORLD_COUNTRIES_GENERAL= 'sensitivity\\World_Countries_(Generalized)_8414823838130214587.gpkg'
VULNERABILITY = "vulnerability.tif"
COUNTRY_MASK = "country_mask.tif"
VULNERABILITY_MEAN = "vulnerability_mean.tif"
MISSING_SENS = 'sensitivity\\missing_count_sensitivity.tif'
MISSING_LACKOF_ADAPT = 'lackof_adapt\\missing_count_lackof_adapt.tif'
MISSING_VULN="missing_count_vulnerability.tif"
VULN_PNG="vulnerability.png"
MISS_PNG="missing_count_vulnerability.png"

In [ ]:
#import packages
import pandas as pd
import geopandas as gpd
import math
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio import warp
from rasterio import features
import geopandas as gpd
from shapely.geometry import box
from rasterio.features import geometry_mask
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, BoundaryNorm
import matplotlib.pyplot as plt
from shapely.ops import unary_union
from rasterio.features import rasterize
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

In [ ]:
#summing up sensitivity and lack of adaptive capacity layers to create vulnerability layer

#Inputs
ref_path = REF
countries_path = WORLD_COUNTRIES_GENERAL

aligned_rasters = [
   SENSITIVITY,
    LACKOF_ADAPT
]

window_size = 2048
dst_nodata = -9999.0

#output paths
out_sum = VULNERABILITY
out_mask = COUNTRY_MASK
out_mean = VULNERABILITY_MEAN

#load reference grid
with rasterio.open(ref_path) as ref:
    ref_meta = ref.meta.copy()
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height


# Windowed country rasterization function to build country mask
def build_country_mask():
    gdf = gpd.read_file(countries_path).to_crs(ref_crs)

    # Keep only valid geometries and drop Antarctica
    gdf = gdf[gdf["COUNTRY"] != "Antarctica"].copy()
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf["geometry"] = gdf.geometry.buffer(0)
    gdf = gdf[gdf.is_valid].copy()
    gdf = gdf[gdf.geometry.notnull()].copy()

    # spatial index for fast window intersection
    sindex = gdf.sindex

    #prepare output raster metadata for country mask
    mask_meta = ref_meta.copy()
    mask_meta.update(
        dtype="uint8",
        count=1,
        nodata=0,
        compress="deflate",
        predictor=2,
        tiled=True,
        blockxsize=256,
        blockysize=256
    )

    #start rasterization of country mask
    with rasterio.open(out_mask, "w", **mask_meta) as dst:
        n_rows = math.ceil(ref_height / window_size)
        n_cols = math.ceil(ref_width / window_size)

        for row in range(n_rows):
            for col in range(n_cols):
                x_off = col * window_size
                y_off = row * window_size
                w = min(window_size, ref_width - x_off)
                h = min(window_size, ref_height - y_off)
                window = Window(x_off, y_off, w, h)

                # window bounds in ref CRS
                win_bounds = rasterio.windows.bounds(window, ref_transform)
                win_geom = box(*win_bounds)

                px = max(abs(ref_transform.a), abs(ref_transform.e))
                win_geom = box(*win_bounds).buffer(px)


                # finds polygons that intersect with this window
                cand_idx = list(sindex.intersection(win_geom.bounds))
                if not cand_idx:
                    # no countries in this window: write 0
                    dst.write(np.zeros((h, w), dtype=np.uint8), 1, window=window)
                    continue

                sub = gdf.iloc[cand_idx]
                sub = sub[sub.intersects(win_geom)]
                if sub.empty:
                    dst.write(np.zeros((h, w), dtype=np.uint8), 1, window=window)
                    continue

                # transform for this window
                win_transform = rasterio.windows.transform(window, ref_transform)
                shapes = ((geom, 1) for geom in sub.geometry)
                burned = features.rasterize(
                    shapes=shapes,
                    out_shape=(h, w),
                    transform=win_transform,
                    fill=0,
                    all_touched=False, 
                    dtype="uint8",
                )
                dst.write(burned, 1, window=window)

    print(f"{out_mask}")

# define function to compute vulnerability
def compute_vulnerability():
    n_vars = len(aligned_rasters)

    #prepare output raster metadata for vulnerability as sum of sensitivity and lack of adaptive capacity
    sum_meta = ref_meta.copy()
    sum_meta.update(
        dtype="float32",
        count=1,
        nodata=dst_nodata,
        compress="deflate",
        predictor=2,
        tiled=True,
        blockxsize=256,
        blockysize=256
    )

    #prepare mean raster to have normalized vulnerability score between 0 and 1 
    mean_meta = ref_meta.copy()
    mean_meta.update(
        dtype="float32",
        count=1,
        nodata=dst_nodata,   
        compress="deflate",
        predictor=2,
        tiled=True,
        blockxsize=256,
        blockysize=256
    )

    # open both rasters once in loop
    srcs = [rasterio.open(p) for p in aligned_rasters]
    #open country mask
    msk = rasterio.open(out_mask)

    try:
        with rasterio.open(out_sum, "w", **sum_meta) as dst_sum, \
             rasterio.open(out_mean, "w", **mean_meta) as dst_mean:


             n_rows = math.ceil(ref_height / window_size)
             n_cols = math.ceil(ref_width / window_size)

             for row in range(n_rows):
                for col in range(n_cols):
                    x_off = col * window_size
                    y_off = row * window_size
                    w = min(window_size, ref_width - x_off)
                    h = min(window_size, ref_height - y_off)
                    window = Window(x_off, y_off, w, h)

                    country = msk.read(1, window=window).astype(bool)  

                    # prepare accumulators
                    sum_arr = np.zeros((h, w), dtype=np.float32)
                    known = np.zeros((h, w), dtype=np.uint8)

                    # For each variable: add where valid and inside countries
                    for s in srcs:
                        a = s.read(1, window=window)

                        # Valid data = not nodata 
                        valid = np.isfinite(a) & (a != dst_nodata)

                        # only sum inside countries
                        use = country & valid
                        if np.any(use):
                            sum_arr[use] += a[use].astype(np.float32)
                            known[use] += 1


                    # Build outputs with following conditions:
                        # outside countries: mark as nodata in sum
                        # inside countries: * if known==0 => true nodata in sum
                                            #* else => sum is valid
                                            # pixel is valid if at least one raster has data
                    has_data = known > 0

                    out_sum_arr = np.full((h, w), dst_nodata, dtype=np.float32)
                    out_mean_arr = np.full((h, w), dst_nodata, dtype=np.float32)

                    
                    # Write outputs only where raster data exists
                    out_sum_arr[has_data] = sum_arr[has_data]
                    out_mean_arr[has_data] = sum_arr[has_data] / known[has_data].astype(np.float32)


                    dst_sum.write(out_sum_arr, 1, window=window)
                    dst_mean.write(out_mean_arr, 1, window=window)

                    
        print(f"{out_sum}")
        print(f"{out_mean}")



    finally:
        for s in srcs:
            s.close()
        msk.close()


In [ ]:
#run functions
build_country_mask()
compute_vulnerability()

In [ ]:
#compute missing count and completenesse for overall vulnerability
#adding the missing count layers of single dimensions

#inputs
missing_sens_path = MISSING_SENS
missing_lack_path = MISSING_LACKOF_ADAPT
country_mask_path = COUNTRY_MASK

#outputs
out_missing_total = MISSING_VULN

#define total variables count
N_SENS = 6
N_LACK = 4
N_TOTAL = N_SENS + N_LACK

#open input rasters
with rasterio.open(missing_sens_path) as ms, \
     rasterio.open(missing_lack_path) as ml, \
     rasterio.open(country_mask_path) as cm:

    #load and update meta for rasters
    meta_missing = ms.meta.copy()
    meta_missing.update(dtype="uint8", count=1, nodata=255, compress="deflate", tiled=True)

    meta_comp = ms.meta.copy()
    meta_comp.update(dtype="float32", count=1, nodata=-9999.0, compress="deflate", tiled=True)
    #open output rasters
    with rasterio.open(out_missing_total, "w", **meta_missing) as dst_miss:
         
        #windowed processing
        for block_index, window in ms.block_windows(1):
            sens = ms.read(1, window=window)
            lack = ml.read(1, window=window)
            country = cm.read(1, window=window).astype(bool)

            # valid where inside country and both layers have data
            valid = country & (sens != 255) & (lack != 255)

            missing_total = np.full(sens.shape, 255, dtype=np.uint8)

            missing_total[valid] = (sens[valid].astype(np.uint16) + lack[valid].astype(np.uint16)).astype(np.uint8)

            known_total = (N_TOTAL - missing_total[valid]).astype(np.float32)
            
            #write output
            dst_miss.write(missing_total, 1, window=window)

print(out_missing_total)


In [ ]:
#plot vulnerability

#paths
raster_path = VULNERABILITY_MEAN
countries_path = WORLD_COUNTRIES_GENERAL
out_png = VULN_PNG

#define resolution
fig_width_in = 14     
dpi = 200
target_width_px = int(fig_width_in * dpi)

#open raster
with rasterio.open(raster_path) as src:
    bounds = src.bounds
    crs = src.crs
    nodata_val = src.nodata

    scale = target_width_px / src.width
    out_w = int(src.width * scale)
    out_h = int(src.height * scale)

    arr = src.read(
        1,
        out_shape=(out_h, out_w),
        resampling=Resampling.nearest
    ).astype("float32")

    transform = src.transform * src.transform.scale(
        src.width / out_w,
        src.height / out_h
    )

#load countries and drop antarctica
world = gpd.read_file(countries_path).to_crs(crs)
world = world[world["COUNTRY"] != "Antarctica"]

#create country mask to define relevant areas of indicator (inside vs outside of countries)
country_mask = geometry_mask(
    geometries=world.geometry,
    transform=transform,
    invert=True,
    out_shape=arr.shape
)

#detect nodata
nodata = ~np.isfinite(arr)
if nodata_val is not None and np.isfinite(nodata_val):
    nodata |= (arr == float(nodata_val))

#mask pixels outside of countries
masked = np.ma.masked_where((~country_mask) | nodata, arr)

#define transparency of country mask
alpha = np.where(country_mask, 1.0, 0.0)

#color scaling
vmin = float(masked.min())
vmax = float(masked.max())

#Build colormap and set masked color to white
nodata_color = "#FFFFFF"
cmap = LinearSegmentedColormap.from_list(
    "vulnerability_red_grad",
    ["#FFF1F2", "#FECACA","#F87171","#DC2626","#7F1D1D"]
)

cmap.set_bad(color=nodata_color)

#create figure
fig, ax = plt.subplots(figsize=(14, 7), dpi=200)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

world.plot(
    ax=ax,
    facecolor="#F6F7F9",
    edgecolor="#FFFFFF",
    linewidth=0.35,
    zorder=1
)

raster = ax.imshow(
    masked,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    interpolation="nearest",
    alpha=alpha,
    zorder=2
)

# Plot country boundaries on top
world.boundary.plot(ax=ax, color="#695C5A", linewidth=0.3, zorder=2)

#set title
ax.set_title(
    "Socio-Ecological Vulnerability to Mining",
    fontsize=18,
    fontweight="semibold",
    pad=14
)
#hide axis
ax.set_axis_off()

#add colorbar
cbar = plt.colorbar(raster, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label(
    "Vulnerability Score",
    fontsize=11,
    color="#2B2F36"
)
cbar.ax.tick_params(labelsize=10, colors="#2B2F36")
cbar.outline.set_edgecolor("#E3E6EA")
cbar.outline.set_linewidth(1.0)
cbar.ax.set_facecolor("white")

#save and show plot
plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()


In [ ]:
#plot count of missing variables in vulnerability

#paths
raster_path = MISSING_VULN
countries_path = WORLD_COUNTRIES_GENERAL
out_png = MISS_PNG

#define resolution
fig_width_in = 14     
dpi = 200
target_width_px = int(fig_width_in * dpi)

#open raster
with rasterio.open(raster_path) as src:
    bounds = src.bounds
    crs = src.crs
    nodata_val = src.nodata

    scale = target_width_px / src.width
    out_w = int(src.width * scale)
    out_h = int(src.height * scale)

    arr = src.read(
        1,
        out_shape=(out_h, out_w),
        resampling=Resampling.nearest
    ).astype("int32")

    transform = src.transform * src.transform.scale(
        src.width / out_w,
        src.height / out_h
    )

#load countries and drop antarctica
world = gpd.read_file(countries_path).to_crs(crs)
world = world[world["COUNTRY"] != "Antarctica"]

#create country mask to define relevant areas of indicator (inside vs outside of countries)
country_mask = geometry_mask(
    geometries=world.geometry,
    transform=transform,
    invert=True,
    out_shape=arr.shape
)

#detect nodata
nodata = ~np.isfinite(arr)
if nodata_val is not None and np.isfinite(nodata_val):
    nodata |= (arr == float(nodata_val))

#mask pixels outside of countries
masked = np.ma.masked_where((~country_mask) | nodata, arr)

#define transparency of country mask
alpha = np.where(country_mask, 1.0, 0.0)

#discrete colormap
vals = masked.compressed()
levels = np.unique(vals).astype(int)

# base palette
base = LinearSegmentedColormap.from_list(
    "sensitivity_red_grad",
    ["#FFF1F2", "#FECACA", "#F87171", "#DC2626", "#7F1D1D"]
)

# create exactly as many colors as levels
colors = base(np.linspace(0, 1, len(levels)))
cmap = ListedColormap(colors)

nodata_color = "#FFFFFF"
cmap.set_bad(color=nodata_color)

# boundaries so each integer gets one color
bounds_disc = np.concatenate(([levels[0] - 0.5], levels + 0.5))
norm = BoundaryNorm(bounds_disc, cmap.N)

#create figure
fig, ax = plt.subplots(figsize=(14, 7), dpi=200)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

world.plot(
    ax=ax,
    facecolor="#F6F7F9",
    edgecolor="#FFFFFF",
    linewidth=0.35,
    zorder=1
)

raster = ax.imshow(
    masked,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    interpolation="nearest",
    alpha=alpha,
    zorder=2
)

# Plot country boundaries on top
world.boundary.plot(ax=ax, color="#695C5A", linewidth=0.3, zorder=2)

#set title
ax.set_title(
    "Count of missing variables: Socio-Ecological Vulnerability",
    fontsize=18,
    fontweight="semibold",
    pad=14
)
#hide axis
ax.set_axis_off()

#discrete colorbar 

cbar = plt.colorbar(
    raster,
    ax=ax,
    fraction=0.03,
    pad=0.02,
    ticks=levels,
    boundaries=bounds_disc,
    spacing="proportional"
)

cbar.set_ticklabels([str(v) for v in levels])
cbar.set_label("Count", fontsize=11, color="#2B2F36")
cbar.ax.tick_params(labelsize=10, colors="#2B2F36")
cbar.outline.set_edgecolor("#E3E6EA")
cbar.outline.set_linewidth(1.0)
cbar.ax.set_facecolor("white")

#save and show plot
plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
